## **Laboratorio: Uso de la API de un LLM con OpenAI**

En este laboratorio aprenderás a interactuar directamente con un modelo de lenguaje de gran escala (LLM) a través de la **API de OpenAI** usando la librería oficial de Python. Este laboratorio conecta con los conceptos vistos en clase: qué información recibe el modelo, cómo se controla la generación de texto, técnicas de prompt engineering, los fallos típicos de los LLMs y cómo evaluarlos de forma básica.


### ¿Qué practicarás?

Conectando con los contenidos de la asignatura, en este laboratorio practicarás:

- Cómo enviar texto a un LLM y entender la estructura de la respuesta (qué recibe el modelo)
- Cómo controlar los parámetros de generación (`temperature`, `max_output_tokens`, `top_p`)
- Cómo recibir la respuesta en tiempo real mediante *streaming*
- Técnicas de prompt engineering: system prompts, few-shot y chain-of-thought
- Cómo obtener salidas estructuradas en JSON y procesarlas con pandas
- Cómo identificar fallos comunes (alucinaciones y sesgos)
- Cómo evaluar respuestas del modelo de forma básica y automática


### Objetivos

Al finalizar este laboratorio, serás capaz de:

- Instalar y configurar la librería `openai` y autenticarte con tu clave de API
- Realizar llamadas a la API de OpenAI usando la **Responses API** y entender la estructura de la respuesta
- Ajustar parámetros de generación como `temperature`, `max_output_tokens` y `top_p` para controlar el estilo de las respuestas
- Aplicar técnicas de prompt engineering (system prompts, few-shot prompting, chain-of-thought) para mejorar la calidad de las respuestas
- Evaluar respuestas del modelo usando métricas básicas y la técnica de LLM-as-judge


> **Estructura de clases**
>
> | Secciones | Clase |
> |-----------|-------|
> | 1 — Configuracion | Clase 1 |
> | 2 — Primera llamada | Clase 1 |
> | 3 — Parametros de generacion | Clase 1 |
> | 4 — Streaming | Clase 1 |
> | 5 — Ingenieria de Prompts | Clase 1 |
> | 6 — Salida estructurada JSON | **Clase 2** |
> | 7 — Fallos comunes | **Clase 2** |
> | 8 — Evaluacion basica | **Clase 2** |

## Sección 1: Configuración del entorno


**Celda 1: Instalación de la librería OpenAI**


In [40]:
!pip install --upgrade openai



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Instalamos la librería oficial de OpenAI para Python. Esta librería nos permite interactuar con los modelos de OpenAI desde nuestro código sin necesidad de hacer peticiones HTTP manualmente.


**Celda 2: Configuración de la clave de API**


In [ ]:
OPENAI_API_KEY = "sk-..."  # ← Reemplaza con tu clave real


La clave de API es el mecanismo de autenticación que identifica tu cuenta ante los servidores de OpenAI. **Nunca compartas tu clave de API** ni la subas a repositorios públicos. En proyectos reales se recomienda guardarla en una variable de entorno o en un archivo `.env`.


## Sección 2: Primera llamada a la API


**Celda 3: Primera llamada directa a la API**


In [42]:
from openai import OpenAI

# Crear el cliente con la clave de API
client = OpenAI(api_key=OPENAI_API_KEY)

# Realizar la primera llamada a la API
response = client.responses.create(
    model="gpt-4o-mini",
    input="Explica qué es un LLM en 3 bullets."
)

# Imprimir la respuesta principal
print("=== Respuesta del modelo ===")
print(response.output_text)

# Inspeccionar el tipo del objeto de respuesta
print("=== Tipo del objeto de respuesta ===")
print(type(response))

# Ver información de uso de tokens
print("=== Uso de tokens ===")
print(response.usage)


=== Respuesta del modelo ===
- **Definición**: Un LLM (Large Language Model) es un modelo de inteligencia artificial diseñado para comprender, generar y manipular texto en lenguaje humano, entrenado con vastas cantidades de datos textuales.

- **Entrenamiento**: Utiliza técnicas de aprendizaje profundo, particularmente redes neuronales transformadoras, para aprender patrones y contextos en el lenguaje, lo que le permite generar respuestas coherentes y relevantes.

- **Aplicaciones**: Se utiliza en diversas áreas, como chatbots, generación de contenido, traducción automática y análisis de sentimientos, entre otros, facilitando la interacción entre humanos y máquinas.
=== Tipo del objeto de respuesta ===
<class 'openai.types.responses.response.Response'>
=== Uso de tokens ===
ResponseUsage(input_tokens=19, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=128, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=147)


Aquí vemos la estructura básica de una llamada a la API:
- `client.responses.create()` es el método principal de la **Responses API**
- `model` especifica qué modelo usar (`gpt-4o-mini` es el más económico)
- `input` es el texto que enviamos al modelo
- `response.output_text` contiene el texto generado
- `response.usage` muestra cuántos tokens se consumieron (importante para controlar costes)


**Celda 4: Función reutilizable `preguntar()`**


In [43]:
def preguntar(pregunta, modelo="gpt-4o-mini"):
    """Envía una pregunta al modelo y devuelve el texto de la respuesta."""
    response = client.responses.create(
        model=modelo,
        input=pregunta
    )
    return response.output_text

# Probar la función
resultado = preguntar("¿Cuál es la capital de Francia?")
print(resultado)


La capital de Francia es París.


Envolver la llamada a la API en una función reutilizable es una buena práctica de programación. Nos permite llamar al modelo con una sola línea de código en el resto del notebook, sin repetir la lógica de inicialización. El parámetro `modelo` con valor por defecto nos da flexibilidad para cambiar de modelo fácilmente.


## Sección 3: Parámetros de Generación


**Celda 5: Efecto de `temperature`**


In [44]:
prompt_temp = "¿Cómo se llama el protagonista de una aventura épica?"

print("=" * 60)
print("TEMPERATURE = 0.0 (determinista, siempre igual)")
print("=" * 60)
for i in range(3):
    respuesta = client.responses.create(
        model="gpt-4o-mini",
        input=prompt_temp,
        temperature=0.0
    )
    print(f"Intento {i+1}: {respuesta.output_text.strip()}")

print()
print("=" * 60)
print("TEMPERATURE = 1.5 (creativa, muy variada)")
print("=" * 60)
for i in range(3):
    respuesta = client.responses.create(
        model="gpt-4o-mini",
        input=prompt_temp,
        temperature=1.5
    )
    print(f"Intento {i+1}: {respuesta.output_text.strip()}")


TEMPERATURE = 0.0 (determinista, siempre igual)
Intento 1: El protagonista de una aventura épica suele llamarse "héroe" o "protagonista". Sin embargo, el nombre específico puede variar según la historia. Por ejemplo, en "El Señor de los Anillos", el protagonista es Frodo Bolsón, mientras que en "La Odisea" es Odiseo. Si tienes una historia en mente, puedo ayudarte a identificar al protagonista.
Intento 2: El protagonista de una aventura épica suele llamarse "héroe" o "héroe épico". Sin embargo, el nombre específico puede variar según la historia. Por ejemplo, en "El Señor de los Anillos", el protagonista es Frodo Bolsón, mientras que en "La Odisea" es Odiseo (Ulises). Si tienes una historia en mente, puedo ayudarte a identificar al protagonista.
Intento 3: El protagonista de una aventura épica suele llamarse "héroe" o "héroe épico". Sin embargo, el nombre específico puede variar según la historia. Por ejemplo, en "El Señor de los Anillos", el protagonista es Frodo Bolsón, mientras que 

La **temperatura** controla la aleatoriedad en la generación de texto:
- `temperature=0.0`: El modelo siempre elige el token más probable. Respuestas muy consistentes y repetibles. Ideal para tareas donde necesitas exactitud (clasificación, código, matemáticas).
- `temperature=1.5`: El modelo explora tokens menos probables. Respuestas más creativas y variadas, pero pueden volverse incoherentes. Útil para tareas creativas.

Observa cómo con temperatura 0.0 las 3 respuestas son prácticamente idénticas, mientras que con 1.5 cada una es diferente.


**Celda 6: Efecto de `max_output_tokens`**


In [45]:
prompt_tokens = "Describe el ciclo del agua"

print("=" * 60)
print("MAX_OUTPUT_TOKENS = 20 (respuesta muy corta)")
print("=" * 60)
respuesta_corta = client.responses.create(
    model="gpt-4o-mini",
    input=prompt_tokens,
    max_output_tokens=20
)
print(respuesta_corta.output_text)
print(f"Tokens usados: {respuesta_corta.usage.output_tokens}")

print()
print("=" * 60)
print("MAX_OUTPUT_TOKENS = 200 (respuesta completa)")
print("=" * 60)
respuesta_larga = client.responses.create(
    model="gpt-4o-mini",
    input=prompt_tokens,
    max_output_tokens=200
)
print(respuesta_larga.output_text)
print(f"Tokens usados: {respuesta_larga.usage.output_tokens}")


MAX_OUTPUT_TOKENS = 20 (respuesta muy corta)


El ciclo del agua, también conocido como ciclo hidrológico, es el proceso continuo mediante el cual
Tokens usados: 20

MAX_OUTPUT_TOKENS = 200 (respuesta completa)
El ciclo del agua, también conocido como ciclo hidrológico, es el proceso continuo de movimiento del agua en la Tierra. Este ciclo está compuesto por varias etapas clave:

1. **Evaporación**: El agua de océanos, ríos, lagos y otras fuentes se calienta por el sol y se transforma en vapor de agua.

2. **Transpiración**: Las plantas también contribuyen al ciclo al liberar vapor de agua a la atmósfera a través de un proceso llamado transpiración.

3. **Condensación**: El vapor de agua en la atmósfera se enfría y se convierte nuevamente en líquido, formando nubes.

4. **Precipitación**: Cuando las gotas de agua en las nubes se agrupan y se vuelven lo suficientemente grandes, caen a la Tierra en forma de lluvia, nieve, granizo o aguanieve.

5. **Infiltración y escorrentía**: Parte del agua que
Tokens usados: 200


`max_output_tokens` limita el número máximo de tokens que el modelo puede generar en la respuesta. Un **token** es aproximadamente 0.75 palabras en inglés (o 0.6 palabras en español). Controlar este parámetro es fundamental para:
- **Gestionar costes**: Menos tokens = menor coste
- **Controlar el formato**: Forzar respuestas concisas
- **Evitar respuestas infinitas**: El modelo se detiene cuando alcanza el límite

Nota: Con 20 tokens la respuesta queda cortada a mitad de frase, lo que ilustra que el límite se aplica de forma brusca.


**Celda 7: Efecto de `top_p`**


In [46]:
prompt_topp = "Dame 3 ideas para un proyecto de IA"

print("=" * 60)
print("TOP_P = 0.1 (solo los tokens más probables)")
print("=" * 60)
respuesta_topp_bajo = client.responses.create(
    model="gpt-4o-mini",
    input=prompt_topp,
    top_p=0.1
)
print(respuesta_topp_bajo.output_text)

print()
print("=" * 60)
print("TOP_P = 1.0 (todos los tokens considerados)")
print("=" * 60)
respuesta_topp_alto = client.responses.create(
    model="gpt-4o-mini",
    input=prompt_topp,
    top_p=1.0
)
print(respuesta_topp_alto.output_text)


TOP_P = 0.1 (solo los tokens más probables)
¡Claro! Aquí tienes tres ideas para un proyecto de inteligencia artificial:

1. **Asistente Virtual Personalizado**:
   - **Descripción**: Desarrolla un asistente virtual que aprenda de las preferencias y hábitos del usuario. Puede ayudar en la gestión del tiempo, recordatorios, recomendaciones de contenido (libros, películas, música) y hasta en la planificación de comidas.
   - **Tecnologías**: Procesamiento de lenguaje natural (NLP), aprendizaje automático, integración con APIs de calendarios y servicios de streaming.

2. **Sistema de Detección de Fake News**:
   - **Descripción**: Crea un sistema que analice artículos de noticias y determine su veracidad. Utiliza técnicas de procesamiento de lenguaje natural para evaluar el contenido y comparar con fuentes confiables.
   - **Tecnologías**: NLP, análisis de sentimientos, aprendizaje supervisado con un conjunto de datos de noticias verificadas y no verificadas.

3. **Plataforma de Recomendac

`top_p` (también llamado **nucleus sampling**) es una alternativa a `temperature` para controlar la aleatoriedad:
- `top_p=0.1`: Solo considera el 10% de la masa de probabilidad acumulada → respuestas más conservadoras y predecibles.
- `top_p=1.0`: Considera el 100% de los tokens posibles → mayor variedad.

**Regla general**: No se recomienda cambiar `temperature` y `top_p` al mismo tiempo. Elige uno de los dos para controlar la aleatoriedad.


## Sección 4: Streaming de Respuestas


**Celda 8: Streaming básico**


In [47]:
from openai.types.responses import ResponseTextDeltaEvent

print("Respuesta en streaming (los tokens aparecen conforme se generan):")
print("-" * 60)

with client.responses.stream(
    model="gpt-4o-mini",
    input="Escribe un poema corto sobre el mar"
) as stream:
    for event in stream:
        if isinstance(event, ResponseTextDeltaEvent):
            print(event.delta, end="", flush=True)

print()  # Nueva linea al finalizar


Respuesta en streaming (los tokens aparecen conforme se generan):
------------------------------------------------------------
Olas que susurran sueños,  
bajo el cielo de azur,  
las caracolas cuentan cuentos  
del viaje sin fin de la luz.

Brillan, danzan en la arena,  
reflejos de un mundo en paz,  
el mar, un abrazo eterno,  
donde las almas saben soñar.


El **streaming** permite recibir la respuesta del modelo token a token, en lugar de esperar a que esté completa. Esto mejora significativamente la experiencia de usuario en aplicaciones interactivas (como ChatGPT), ya que el usuario ve el texto aparecer progresivamente en lugar de esperar varios segundos por la respuesta completa.

- `client.responses.stream()` devuelve un gestor de contexto (`with`)
- `ResponseTextDeltaEvent` es el evento que contiene cada fragmento de texto generado
- `flush=True` fuerza que el texto se muestre inmediatamente sin buffering


**Celda 9: Streaming con recolección de texto**


In [48]:
from openai.types.responses import ResponseTextDeltaEvent

fragmentos = []

print("Recibiendo respuesta en streaming...")
print("-" * 60)

with client.responses.stream(
    model="gpt-4o-mini",
    input="Explica en 3 parrafos la importancia de la inteligencia artificial"
) as stream:
    for event in stream:
        if isinstance(event, ResponseTextDeltaEvent):
            fragmentos.append(event.delta)          # Guardar cada fragmento
            print(event.delta, end="", flush=True)  # Mostrar en tiempo real

print()  # Nueva linea

# Unir todos los fragmentos y calcular estadisticas
texto_completo = "".join(fragmentos)
print("" + "=" * 60)
print(f"Total de fragmentos recibidos: {len(fragmentos)}")
print(f"Total de caracteres: {len(texto_completo)}")
print(f"Total de palabras aproximadas: {len(texto_completo.split())}")


Recibiendo respuesta en streaming...
------------------------------------------------------------
La inteligencia artificial (IA) ha revolucionado numerosos sectores, desde la salud hasta la educación y la industria. Su capacidad para procesar grandes volúmenes de datos permite identificar patrones y realizar predicciones con una precisión que superaría la capacidad humana. En medicina, por ejemplo, la IA se utiliza para diagnosticar enfermedades a partir de imágenes médicas o datos genéticos, lo que no solo puede acelerar el proceso de diagnóstico, sino también mejorar la personalización de tratamientos. Esta transformación en el ámbito de la salud es solo un ejemplo de cómo la IA puede desempeñar un papel crítico en la mejora de la calidad de vida y en la eficiencia de servicios esenciales.

En el mundo empresarial, la IA ha permitido optimizar procesos operativos y estratégicos. Herramientas impulsadas por IA, como chatbots y sistemas de recomendación, mejoran la experiencia del cli

En aplicaciones reales, a menudo necesitas tanto mostrar el texto en tiempo real **como** conservarlo para procesarlo después. La solución es guardar cada fragmento en una lista (`fragmentos`) mientras se imprime, y luego unir todos los fragmentos con `"".join()`. Este es un patrón muy común en aplicaciones de chat con LLMs.


## Sección 5: Ingeniería de Prompts


**Celda 10: System prompt**


In [49]:
respuesta_system = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {
            "role": "system",
            "content": "Eres un asistente experto en Python que responde SIEMPRE en formato de lista numerada."
        },
        {
            "role": "user",
            "content": "¿Cómo se leen datos de un CSV en Python?"
        }
    ]
)

print(respuesta_system.output_text)


Para leer datos de un archivo CSV en Python, puedes seguir estas instrucciones:

1. **Importar la biblioteca**:
   - Utiliza la biblioteca `csv` o `pandas`.

2. **Usando la biblioteca `csv`**:
   1. Abre el archivo CSV utilizando `open()`.
   2. Crea un objeto lector usando `csv.reader()`.
   3. Itera sobre las filas del archivo.

   Ejemplo:
   ```python
   import csv

   with open('archivo.csv', mode='r') as file:
       lector = csv.reader(file)
       for fila in lector:
           print(fila)
   ```

3. **Usando la biblioteca `pandas`**:
   1. Instala `pandas` si no lo tienes ya instalado (`pip install pandas`).
   2. Usa `pd.read_csv()` para leer el archivo CSV.

   Ejemplo:
   ```python
   import pandas as pd

   df = pd.read_csv('archivo.csv')
   print(df)
   ```

4. **Especificar delimitadores** (si es diferente a coma):
   - En `csv`, puedes usar el parámetro `delimiter`.
   - En `pandas`, puedes usar el parámetro `sep`.

5. **Manejo de encabezados**:
   - En `csv`, los encab

El **system prompt** es una instrucción especial que define el comportamiento, personalidad y restricciones del modelo para toda la conversación. Se especifica con `"role": "system"` y aparece antes del mensaje del usuario. Es una de las técnicas más poderosas del prompt engineering porque:
- Define el **rol** del modelo (experto en Python, asistente médico, etc.)
- Establece **restricciones de formato** (lista numerada, JSON, etc.)
- Controla el **tono** y el **estilo** de las respuestas
- Puede incluir **contexto** relevante para toda la sesión


**Celda 11: Few-shot prompting**


In [50]:
# Few-shot: clasificación de reseñas de productos
respuesta_few_shot = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {
            "role": "system",
            "content": "Clasifica reseñas de productos como: positivo, negativo o neutro. Responde SOLO con una palabra."
        },
        # Ejemplo 1
        {
            "role": "user",
            "content": "Reseña: 'Producto increíble, superó todas mis expectativas. Lo recomiendo al 100%'"
        },
        {
            "role": "assistant",
            "content": "positivo"
        },
        # Ejemplo 2
        {
            "role": "user",
            "content": "Reseña: 'El producto llegó roto y el servicio al cliente no respondió mis mensajes'"
        },
        {
            "role": "assistant",
            "content": "negativo"
        },
        # Ejemplo 3
        {
            "role": "user",
            "content": "Reseña: 'El artículo es tal como se describe en la página web. Entrega en el plazo indicado'"
        },
        {
            "role": "assistant",
            "content": "neutro"
        },
        # Nueva reseña a clasificar
        {
            "role": "user",
            "content": "Reseña: 'Malísimo, se rompió a los dos días de usarlo. Una pérdida de dinero total'"
        }
    ]
)

print(f"Clasificación: {respuesta_few_shot.output_text.strip()}")


Clasificación: negativo


El **few-shot prompting** consiste en proporcionar al modelo ejemplos de entradas y salidas correctas antes de hacer la pregunta real. El modelo aprende el patrón a partir de estos ejemplos sin necesidad de entrenamiento adicional. En este caso:
- Damos 3 ejemplos (3-shot) de reseñas con su clasificación correcta
- El modelo aprende el formato esperado: una sola palabra (`positivo`, `negativo`, `neutro`)
- Esto es mucho más efectivo que simplemente decir "clasifica como positivo/negativo/neutro"


**Celda 12: Chain-of-thought prompting**


In [51]:
problema = """Si tengo 15 manzanas y reparto 3 a cada uno de mis 4 amigos, ¿cuántas me quedan?
Razona paso a paso antes de dar la respuesta final."""

respuesta_cot = client.responses.create(
    model="gpt-4o-mini",
    input=problema
)

print(respuesta_cot.output_text)


Claro, vamos a resolverlo paso a paso.

1. **Cantidad inicial de manzanas**: Tienes 15 manzanas.

2. **Número de amigos**: Tienes 4 amigos.

3. **Manzanas por amigo**: Decides repartir 3 manzanas a cada amigo.

4. **Cálculo de manzanas repartidas**:
   - Manzanas que das a cada amigo: 3
   - Número de amigos: 4
   - Total de manzanas repartidas = 3 manzanas/amigo × 4 amigos = 12 manzanas.

5. **Cálculo de manzanas restantes**:
   - Manzanas iniciales: 15
   - Manzanas repartidas: 12
   - Manzanas que te quedan = 15 - 12 = 3 manzanas.

Por lo tanto, después de repartir las manzanas, te quedan **3 manzanas**.


El **chain-of-thought (CoT)** o razonamiento en cadena es una técnica que mejora drásticamente el rendimiento del modelo en problemas que requieren múltiples pasos lógicos. Al pedirle que "razone paso a paso", el modelo:
1. Descompone el problema en pasos intermedios
2. Resuelve cada paso antes de llegar a la conclusión
3. Reduce la probabilidad de cometer errores en razonamiento aritmético o lógico

Esta técnica es especialmente efectiva para problemas matemáticos, de lógica, código, y cualquier tarea que requiera múltiples pasos de razonamiento.


---
# 🔷 A PARTIR AQUI: CLASE 2 (Tema 18.2 — LLMs Multimodales y en Produccion)
---
Las secciones siguientes se cubren en la segunda clase: salida estructurada para pipelines de datos, fallos tipicos de LLMs (alucinaciones, sesgo) y evaluacion automatica de respuestas.

## Sección 6: Salida Estructurada en JSON


**Celda 13: Obtener JSON del modelo**


In [52]:
import json
import re

texto_a_analizar = "El servicio fue horrible, esperé 2 horas y la comida llegó fría."

prompt_json = f"""Analiza el siguiente texto y devuelve SOLO un JSON válido con estos campos:
- sentimiento: positivo, negativo o neutro
- confianza: número entre 0 y 1
- palabras_clave: lista de exactamente 3 palabras clave

Devuelve SOLO el JSON puro, sin bloques de código markdown.

Texto: '{texto_a_analizar}'"""

respuesta_json = client.responses.create(
    model="gpt-4o-mini",
    input=prompt_json,
    temperature=0.0
)

# Limpiar posibles bloques markdown (```json ... ```) y parsear
texto_json = respuesta_json.output_text.strip()
texto_json = re.sub(r'^```(?:json)?\s*\n?', '', texto_json)
texto_json = re.sub(r'\n?```\s*$', '', texto_json).strip()

print("Respuesta raw del modelo:")
print(texto_json)

datos = json.loads(texto_json)

print("\n=== Datos parseados ===")
print(f"Sentimiento: {datos['sentimiento']}")
print(f"Confianza: {datos['confianza']}")
print(f"Palabras clave: {datos['palabras_clave']}")


Respuesta raw del modelo:
{
  "sentimiento": "negativo",
  "confianza": 0.9,
  "palabras_clave": ["servicio", "horrible", "comida"]
}

=== Datos parseados ===
Sentimiento: negativo
Confianza: 0.9
Palabras clave: ['servicio', 'horrible', 'comida']


Obtener salidas estructuradas en JSON es esencial cuando el LLM forma parte de un pipeline de datos o una aplicación. En lugar de texto libre difícil de procesar, podemos pedir al modelo que devuelva datos estructurados que podemos usar directamente en nuestro código. Puntos clave:
- Especificar claramente los campos y sus tipos en el prompt
- Pedir que devuelva "SOLO" el JSON sin texto adicional
- Usar `temperature=0.0` para mayor consistencia
- Siempre usar `json.loads()` para parsear la respuesta (puede fallar → usar try/except en producción)


**Celda 14: Análisis de múltiples ítems con pandas**


In [53]:
import pandas as pd
import re

resenas = [
    "La pizza estaba deliciosa y el ambiente era muy acogedor. Definitivamente volveré.",
    "Tuve que esperar 45 minutos para ser atendido y la cuenta tenía errores.",
    "El local está bien ubicado. La comida es normal, ni destacable ni mala."
]

resultados = []

for i, resena in enumerate(resenas):
    print(f"Analizando reseña {i+1}...")

    prompt = f"""Analiza la siguiente reseña y devuelve SOLO un JSON válido con:
- sentimiento: positivo, negativo o neutro
- confianza: número entre 0 y 1
- palabras_clave: lista de exactamente 3 palabras clave

Devuelve SOLO el JSON puro, sin bloques de código markdown.

Reseña: '{resena}'"""

    respuesta = client.responses.create(
        model="gpt-4o-mini",
        input=prompt,
        temperature=0.0
    )

    texto = respuesta.output_text.strip()
    texto = re.sub(r'^```(?:json)?\s*\n?', '', texto)
    texto = re.sub(r'\n?```\s*$', '', texto).strip()
    datos = json.loads(texto)
    datos["resena"] = resena[:50] + "..."
    resultados.append(datos)

df = pd.DataFrame(resultados)
df.head()


Analizando reseña 1...
Analizando reseña 2...
Analizando reseña 3...


,sentimiento,confianza,palabras_clave,resena
0,positivo,0.9,"[pizza, deliciosa, acogedor]",La pizza estaba deliciosa y el ambiente era mu...
1,negativo,0.8,"[esperar, atendido, errores]",Tuve que esperar 45 minutos para ser atendido ...
2,neutro,0.6,"[ubicado, comida, normal]",El local está bien ubicado. La comida es norma...


Cuando necesitamos analizar múltiples elementos, iteramos sobre ellos y acumulamos los resultados en una lista de diccionarios. Luego convertimos esa lista en un DataFrame de pandas, que nos da acceso a todas las herramientas de análisis de datos: filtrado, agrupación, visualización, exportación a CSV, etc. Este patrón es la base de muchos pipelines de análisis de texto con LLMs.


## Sección 7: Fallos Comunes de los LLMs


**Celda 15: Alucinaciones**


In [54]:
# Pregunta con premisa falsa — el modelo puede inventar detalles
print("=" * 60)
print("PREGUNTA 1: Premisa falsa histórica")
print("=" * 60)
pregunta1 = "¿Qué dijo Napoleón en su famoso discurso de 1802 en Madrid?"
print(f"Pregunta: {pregunta1}")
print("Respuesta del modelo:")
print(preguntar(pregunta1))

print()
print("=" * 60)
print("PREGUNTA 2: Dato nutricional concreto")
print("=" * 60)
pregunta2 = "¿Cuántos gramos de proteína tiene exactamente una manzana grande de 200 gramos?"
print(f"Pregunta: {pregunta2}")
print("Respuesta del modelo:")
print(preguntar(pregunta2))
print()
print("[NOTA: Una manzana de 200g contiene aproximadamente 0.6g de proteína. Verifica si el modelo es preciso o exagera.]")


PREGUNTA 1: Premisa falsa histórica
Pregunta: ¿Qué dijo Napoleón en su famoso discurso de 1802 en Madrid?
Respuesta del modelo:
Napoleón Bonaparte, en su discurso de 1802 en Madrid, no ofreció un discurso formalizado como tal. En ese año, la presencia francesa en España era parte del contexto de la Guerra Peninsular. Sin embargo, se sabe que Napoleón buscaba justificar su intervención en los asuntos españoles y promover sus ideas sobre el orden y la69 paz en Europa.

Si tienes un evento o discurso específico en mente, por favor, proporciona más detalles y estaré encantado de ayudarte.

PREGUNTA 2: Dato nutricional concreto
Pregunta: ¿Cuántos gramos de proteína tiene exactamente una manzana grande de 200 gramos?
Respuesta del modelo:
Una manzana grande de aproximadamente 200 gramos contiene alrededor de 0.5 a 1 gramo de proteína. La cantidad exacta puede variar ligeramente según el tipo de manzana, pero en general, las manzanas no son una fuente significativa de proteína.

[NOTA: Una ma

Las **alucinaciones** son uno de los fallos más importantes de los LLMs: el modelo genera información plausible pero incorrecta con total confianza. Esto ocurre porque los LLMs están entrenados para generar texto coherente, no para verificar la veracidad de los hechos.

**Tipos comunes de alucinaciones:**
- **Eventos históricos inventados**: El modelo puede inventar discursos, fechas o lugares que nunca ocurrieron
- **Datos numéricos inexactos**: Cifras que suenan plausibles pero son incorrectas
- **Referencias bibliográficas falsas**: El modelo puede citar libros o artículos que no existen

**Solución**: Siempre verificar hechos críticos con fuentes externas. En aplicaciones serias, combinar LLMs con sistemas de recuperación de información (RAG).


**Celda 16: Sesgo y limitaciones**


In [55]:
tema = "el uso de redes sociales en adolescentes"

print("=" * 60)
print("PROMPT SESGADO (fuerza respuesta negativa)")
print("=" * 60)
prompt_sesgado = f"Dame 3 razones por las que {tema} es perjudicial"
print(f"Prompt: {prompt_sesgado}")
print("Respuesta:")
print(preguntar(prompt_sesgado))

print()
print("=" * 60)
print("PROMPT NEUTRO (solicita perspectiva equilibrada)")
print("=" * 60)
prompt_neutro = f"Dame 3 aspectos positivos y 3 aspectos negativos de {tema}"
print(f"Prompt: {prompt_neutro}")
print("Respuesta:")
print(preguntar(prompt_neutro))


PROMPT SESGADO (fuerza respuesta negativa)
Prompt: Dame 3 razones por las que el uso de redes sociales en adolescentes es perjudicial
Respuesta:
Claro, aquí tienes tres razones por las que el uso de redes sociales en adolescentes puede ser perjudicial:

1. **Impacto en la salud mental**: El uso excesivo de redes sociales puede contribuir a la ansiedad, depresión y baja autoestima. Compararse constantemente con las imágenes idealizadas de la vida de otros puede hacer que los adolescentes se sientan inadecuados o insatisfechos con su propia vida.

2. **Ciberacoso**: Las redes sociales pueden facilitar el acoso en línea, lo que puede tener consecuencias graves para la salud emocional y mental de los adolescentes. El ciberacoso puede ser persistente y difícil de escapar, afectando su bienestar y su capacidad para socializar.

3. **Distracción y problemas de concentración**: La constante interacción en redes sociales puede distraer a los adolescentes de sus responsabilidades académicas y ot

Los LLMs son muy sensibles al **encuadre del prompt** (framing). Si el prompt ya contiene un sesgo implícito, el modelo lo amplificará en su respuesta. Esto es importante porque:
- El modelo no tiene opiniones propias: responde según lo que se le pregunta
- Un prompt mal formulado puede generar contenido engañoso o parcial
- En aplicaciones de análisis o investigación, es crucial usar prompts neutros y equilibrados

Además de los sesgos introducidos por el prompt, los LLMs pueden heredar sesgos de sus datos de entrenamiento (sesgos de género, culturales, etc.). Es importante ser consciente de estas limitaciones.


## Sección 8: Evaluación Básica


**Celda 17: Evaluación por coincidencia exacta**


In [56]:
def normalizar(texto):
    """Limpia la respuesta para comparación: sin puntuación final ni Unicode especial."""
    import re, unicodedata
    subscripts = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
    return re.sub(r"[.,;:!?]+$", "", texto.strip().translate(subscripts))

# Preguntas con respuestas esperadas
preguntas_respuestas = [
    {
        "pregunta": "¿Cuál es la capital de España? Responde SOLO con el nombre de la ciudad.",
        "esperado": "Madrid"
    },
    {
        "pregunta": "¿Cuántos planetas tiene el sistema solar? Responde SOLO con el número.",
        "esperado": "8"
    },
    {
        "pregunta": "¿En qué año terminó la Segunda Guerra Mundial? Responde SOLO con el año.",
        "esperado": "1945"
    }
]

aciertos = 0
total = len(preguntas_respuestas)

print("=" * 60)
print("EVALUACIÓN POR COINCIDENCIA EXACTA")
print("=" * 60)

for i, qa in enumerate(preguntas_respuestas):
    respuesta_modelo = normalizar(preguntar(qa["pregunta"]))
    correcto = respuesta_modelo == qa["esperado"]
    if correcto:
        aciertos += 1

    print(f"\nPregunta {i+1}: {qa['pregunta']}")
    print(f"  Esperado : '{qa['esperado']}'")
    print(f"  Obtenido : '{respuesta_modelo}'")
    resultado = 'CORRECTO' if correcto else 'INCORRECTO'
    print(f"  Resultado: {resultado}")

print()
print("=" * 60)
print(f"Precisión: {aciertos}/{total} = {aciertos/total*100:.1f}%")
print("=" * 60)


EVALUACIÓN POR COINCIDENCIA EXACTA

Pregunta 1: ¿Cuál es la capital de España? Responde SOLO con el nombre de la ciudad.
  Esperado : 'Madrid'
  Obtenido : 'Madrid'
  Resultado: CORRECTO

Pregunta 2: ¿Cuántos planetas tiene el sistema solar? Responde SOLO con el número.
  Esperado : '8'
  Obtenido : 'Ocho'
  Resultado: INCORRECTO

Pregunta 3: ¿En qué año terminó la Segunda Guerra Mundial? Responde SOLO con el año.
  Esperado : '1945'
  Obtenido : '1945'
  Resultado: CORRECTO

Precisión: 2/3 = 66.7%


La **evaluación por coincidencia exacta** compara la respuesta del modelo directamente con la respuesta esperada usando el operador `==`. Es la métrica más simple pero también la más limitada:
- **Ventajas**: Fácil de implementar, completamente objetiva, rápida
- **Limitaciones**: Cualquier variación en el formato hace que falle (`"Madrid"` ≠ `"madrid"`, `"8"` ≠ `"ocho"`)

Para mejorarla, se pueden aplicar normalizaciones: `.lower()`, `.strip()`, eliminar puntuación. Esta métrica es adecuada para preguntas con respuestas muy cortas y bien definidas.


**Celda 18: LLM-as-judge**


In [57]:
evaluaciones = [
    {
        "pregunta": "¿Qué es el machine learning?",
        "respuesta": "Machine learning es una rama de la IA donde los modelos aprenden patrones a partir de datos para hacer predicciones o decisiones sin ser programados explícitamente."
    },
    {
        "pregunta": "¿Qué es el machine learning?",
        "respuesta": "Es algo de ordenadores."
    },
    {
        "pregunta": "¿Cuáles son las ventajas del aprendizaje supervisado?",
        "respuesta": "El aprendizaje supervisado permite entrenar modelos con datos etiquetados, obteniendo predicciones precisas, es fácil de evaluar con métricas estándar y funciona bien con problemas bien definidos como clasificación y regresión."
    }
]

print("=" * 60)
print("LLM-AS-JUDGE: Evaluación automática con el modelo")
print("=" * 60)

puntuaciones = []

for i, eval_item in enumerate(evaluaciones):
    prompt_juez = f"""Evalúa la siguiente respuesta del 1 al 5 según precisión y claridad.
Responde SOLO con el número (1=muy mala, 5=excelente).

Pregunta: {eval_item['pregunta']}
Respuesta: {eval_item['respuesta']}"""

    puntuacion_raw = client.responses.create(
        model="gpt-4o-mini",
        input=prompt_juez,
        temperature=0.0
    ).output_text.strip()

    puntuaciones.append(int(puntuacion_raw))

    print(f"\nEvaluación {i+1}:")
    print(f"  Pregunta : {eval_item['pregunta']}")
    resp_display = eval_item['respuesta']
    print(f"  Respuesta: {resp_display[:80]}..." if len(resp_display) > 80 else f"  Respuesta: {resp_display}")
    print(f"  Puntuación del juez: {puntuacion_raw}/5")

print()
print(f"Puntuación media: {sum(puntuaciones)/len(puntuaciones):.2f}/5")


LLM-AS-JUDGE: Evaluación automática con el modelo

Evaluación 1:
  Pregunta : ¿Qué es el machine learning?
  Respuesta: Machine learning es una rama de la IA donde los modelos aprenden patrones a part...
  Puntuación del juez: 5/5

Evaluación 2:
  Pregunta : ¿Qué es el machine learning?
  Respuesta: Es algo de ordenadores.
  Puntuación del juez: 1/5

Evaluación 3:
  Pregunta : ¿Cuáles son las ventajas del aprendizaje supervisado?
  Respuesta: El aprendizaje supervisado permite entrenar modelos con datos etiquetados, obten...
  Puntuación del juez: 5/5

Puntuación media: 3.67/5


**LLM-as-judge** (o LLM como juez) es una técnica que usa un LLM para evaluar la calidad de las respuestas de otro LLM (o del mismo). Es especialmente útil cuando:
- Las respuestas son texto libre y no hay una respuesta "correcta" única
- Queremos evaluar criterios subjetivos como claridad, coherencia o utilidad
- La evaluación manual sería demasiado costosa o lenta

**Consideraciones**: El modelo juez puede tener sus propios sesgos. Para evaluaciones más robustas se recomienda usar múltiples jueces, criterios de evaluación detallados, y calibrar con evaluaciones humanas.


**Celda 19: Benchmark básico con pandas**


In [58]:
import pandas as pd

# Benchmark de 5 pares pregunta-respuesta esperada
benchmark = [
    {"pregunta": "¿Cuál es la capital de Alemania? (solo el nombre)", "esperado": "Berlín"},
    {"pregunta": "¿Cuánto es 7 multiplicado por 8? (solo el número)", "esperado": "56"},
    {"pregunta": "¿Quién escribió Don Quijote? (solo el nombre)", "esperado": "Miguel de Cervantes"},
    {"pregunta": "¿Cuál es el símbolo químico del agua? (solo la fórmula)", "esperado": "H2O"},
    {"pregunta": "¿En qué año llegó el hombre a la Luna? (solo el año)", "esperado": "1969"}
]

filas = []

for item in benchmark:
    respuesta = normalizar(preguntar(item["pregunta"]))
    correcto = respuesta == item["esperado"]

    filas.append({
        "Pregunta": item["pregunta"][:50] + "...",
        "Esperado": item["esperado"],
        "Obtenido": respuesta,
        "Correcto": "Sí" if correcto else "No"
    })

df_benchmark = pd.DataFrame(filas)

print("=" * 60)
print("RESULTADOS DEL BENCHMARK")
print("=" * 60)
print(df_benchmark.to_string(index=False))

aciertos_total = (df_benchmark["Correcto"] == "Sí").sum()
print(f"\nPrecisión total: {aciertos_total}/{len(benchmark)} ({aciertos_total/len(benchmark)*100:.0f}%)")


RESULTADOS DEL BENCHMARK
                                             Pregunta            Esperado            Obtenido Correcto
 ¿Cuál es la capital de Alemania? (solo el nombre)...              Berlín              Berlín       Sí
 ¿Cuánto es 7 multiplicado por 8? (solo el número)...                  56                  56       Sí
     ¿Quién escribió Don Quijote? (solo el nombre)... Miguel de Cervantes Miguel de Cervantes       Sí
¿Cuál es el símbolo químico del agua? (solo la fór...                 H2O                 H2O       Sí
¿En qué año llegó el hombre a la Luna? (solo el añ...                1969                1969       Sí

Precisión total: 5/5 (100%)


Un **benchmark** es un conjunto estandarizado de pruebas que nos permite medir el rendimiento del modelo de forma sistemática y repetible. Este patrón es la base de los benchmarks industriales como MMLU, HellaSwag o HumanEval. Las ventajas de usar pandas para mostrar los resultados:
- Vista tabular clara y comparativa
- Fácil de exportar a CSV para análisis posteriores (`df.to_csv()`)
- Se puede extender con más métricas (tiempo de respuesta, tokens usados, etc.)
